<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/circuit_tracer_examples/jspace_workspace_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# J-space workspace readout: think X, say Y

The Jacobian lens reads an internal activation as the tokens it is disposed to make the model say, by transporting the
residual $h_\ell$ at layer $\ell$ into the final-layer basis with the average input-output Jacobian $\bar J_\ell$.
The steering demo (`ct_concept_steering_demo`) uses that space to *write*; this notebook uses it only to *read*.

The question is the one the Jacobian-lens paper asks of latent multi-step reasoning. When a prompt needs an
intermediate it never states, is that intermediate readable in the workspace before the answer is?

- **Say Y:** the answer the model gives (`racket`), read at the answer position layer by layer.
- **Think X:** the unstated bridge entity (`tennis`), which may surface at an earlier position rather than at the
  answer position, so it is read at every position and every fitted layer.

**How typical this prompt is.** It was chosen from the paper's two-hop set because the ordering is clear on it. Over
the 33 items of that set that gemma-3-1b-it answers correctly with single-token entities, the intermediate is readable
(lens top-5, any position) strictly before the answer on 24, the answer comes first or ties on 7, and the
intermediate is never readable on 2. One of the 7 (`the Amazon River ... Portuguese`, via `Brazil`) is the notebook's
second test parameterization, so the counter-case runs too. Rates with a specificity control are in
interpretune#425.

In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
MODEL_FAMILY = "gemma3"
MODEL_VARIANT = "1b_it"
LENS_MODEL_ID = "gemma-3-1b-it"  # directory in neuronpedia/jacobian-lens
# A two-hop fact from the Jacobian-lens paper's own battery: the answer requires an intermediate the prompt never names.
PROMPT = "Fact: The piece of equipment used to strike the ball in the sport played at Wimbledon is a"
THINK_TOKEN = "tennis"  # the unstated intermediate ("think X")
SAY_TOKEN = "racket"  # the answer ("say Y")
RANK_DEPTH = 200  # a token outside the lens top-RANK_DEPTH is reported as not readable
READABLE_TOP_K = 5  # "readable" means inside the lens top-5

In [3]:
# @title Imports { display-mode: "form" }
import tempfile
from contextlib import ExitStack

import torch

import interpretune as it
import interpretune.analysis  # noqa: F401  # ensure op wrappers are registered
from interpretune.analysis.ops.base import AnalysisBatch
from interpretune.analysis.ops.bundled.jlens.jlens_ops import default_jlens_cache_key
from interpretune.harness.sessions import experiment_session
from it_examples.utils.nb_ui_utils import display_jlens_readout_comparison

~/repos/interpretune/.claude/worktrees/jlens-research-639/src/interpretune/hub/trust.py:96: Refusing to execute analysis ops from 'speediedan/jlens_steering_ops': loading it runs code published by that repo inside this process, with your filesystem, network and Hugging Face token. Interpretune does not execute hub-resident code unless you opt in.
  Inspect it first: interpretune.hub.pull('speediedan/jlens_steering_ops') caches the repo without executing anything, so you can read the entrypoint the manifest names.
  Then opt in for this session: os.environ['IT_TRUST_REMOTE_CODE'] = '1' (or export IT_TRUST_REMOTE_CODE=1 before starting the process).
  Pin a revision when you pull so trusted code cannot change under you.
  Details: https://interpretune.org/en/latest/usage/hub_trust_posture.html


## 1: Session

The harness session machinery builds the model from a registered model spec. It is held open across cells with an
`ExitStack` and closed in the last cell.

In [4]:
_stack = ExitStack()
session, module, tokenizer = _stack.enter_context(
    experiment_session(
        tempfile.mkdtemp(), "jspace_workspace_demo", model_family=MODEL_FAMILY, model_variant=MODEL_VARIANT
    )
)
# the HF model under the nnsight wrapper (`_model`; `_module` on older nnsight releases)
hf = getattr(module.model, "_model", None) or getattr(module.model, "_module", None)
device = next(hf.parameters()).device
print(type(hf).__name__, "on", device)

~/repos/interpretune/.claude/worktrees/jlens-research-639/src/interpretune/config/shared.py:271: Could not find an auto-composition for <class 'interpretune.config.module.ITConfig'> that supports all of the following kwargs: {'circuit_tracer_cfg': CircuitTracerConfig(backend='nnsight', model_name='google/gemma-3-1b-it', transcoder_set='mwhanna/gemma-scope-2-1b-it/transcoder_all/width_16k_l0_small_affine', dtype=torch.bfloat16, max_n_logits=10, desired_logit_prob=0.95, batch_size=256, max_feature_nodes=None, offload=None, lazy_encoder=None, lazy_decoder=True, verbose=True, default_node_threshold=0.8, default_edge_threshold=0.98, save_graphs=True, graph_output_dir=None, analysis_target_tokens=None, target_token_ids=None, use_neuronpedia=False, intervention_scale_factor=1.0, intervention_max_influence_norm_scale=False, intervention_sign_aware_scale=True, intervention_value=None, intervention_value_source='top_feature_scores', intervention_constrained_layers=None, intervention_freeze_atten

[INFO] interpretune.utils.logging: Loading ReplacementModel with backend: nnsight


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: NNsight ReplacementModel initialized for Circuit Tracer


~/repos/interpretune/.claude/worktrees/jlens-research-639/src/interpretune/session.py:314: CircuitTracerModule is not interpretune.protocol.BuiltTrainSteppable | interpretune.protocol.BuiltValidationSteppable | interpretune.protocol.BuiltTestSteppable | interpretune.protocol.BuiltPredictSteppable even after composing <class 'interpretune.adapters.circuit_tracer.adapter.CircuitTracerModule'> with <class 'interpretune.session.InterpretunableModule'>. This Interpretune session may not work as intended.
[INFO] interpretune.utils.logging: Preparing data: ExperimentDataModule


[INFO] interpretune.utils.logging: Setting up datamodule: ExperimentDataModule


[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


Gemma3ForCausalLM on cuda:0


## 2: One forward pass, every block output captured

Each lens layer was fitted on its block's **output**, which is also the ops' default cache point
(`default_jlens_cache_key`). One forward pass with a hook on every decoder layer gives all of them.

In [5]:
def _single_token(text):
    ids = [i for i in tokenizer.encode(" " + text, add_special_tokens=False) if tokenizer.decode([i]).strip()]
    if len(ids) != 1:
        raise ValueError(f"{text!r} is {len(ids)} content tokens; each concept needs a single J-space row")
    return ids[0]


think_id, say_id = _single_token(THINK_TOKEN), _single_token(SAY_TOKEN)
input_ids = tokenizer((tokenizer.bos_token or "") + PROMPT, return_tensors="pt", add_special_tokens=False).input_ids
input_ids = input_ids.to(device)
prompt_tokens = [tokenizer.decode([i]) for i in input_ids[0].tolist()]

acts = {}
_hooks = [
    layer.register_forward_hook(
        lambda _m, _i, out, idx=idx: acts.__setitem__(idx, (out[0] if isinstance(out, tuple) else out).detach())
    )
    for idx, layer in enumerate(hf.model.layers)
]
try:
    with torch.no_grad():
        logits = hf(input_ids).logits
finally:
    for _h in _hooks:
        _h.remove()
print("prompt:", PROMPT)
print("model answers:", repr(tokenizer.decode([int(logits[0, -1].argmax())])))

prompt: Fact: The piece of equipment used to strike the ball in the sport played at Wimbledon is a
model answers: ' racket'


## 3: What the workspace says at the answer position, layer by layer

`it.jlens_read` ranks the vocabulary by the lens readout. Each concept's rank comes from the top-`RANK_DEPTH` list
(0 is top-1).

In [6]:
def read(layer, positions):
    key = default_jlens_cache_key(layer)
    return it.jlens_read(
        module,
        AnalysisBatch(cache={key: acts[layer]}),
        None,
        0,
        jlens_layer=layer,
        jlens_model_id=LENS_MODEL_ID,
        jlens_positions=positions,
        jlens_top_k=RANK_DEPTH,
    )


def rank_in(ids_row, token_id):
    hits = (ids_row == token_id).nonzero()
    return int(hits[0]) if len(hits) else None


answer_pos = input_ids.shape[1] - 1
rows = []
for layer in sorted(acts):
    try:
        out = read(layer, [answer_pos])
    except ValueError:  # a layer the lens was not fitted at is refused by name; skip it
        continue
    ids_row = torch.as_tensor(out["jlens_top_token_ids"])[0, 0]
    top = [tokenizer.decode([int(t)]) for t in ids_row[:5]]
    rows.append((layer, top, rank_in(ids_row, think_id), rank_in(ids_row, say_id)))


def _fmt(r):
    return "-" if r is None else str(r)


print(f"{'layer':>5} | {'top-5 at the answer position':<60} | {THINK_TOKEN} rank | {SAY_TOKEN} rank")
for layer, top, r_think, r_say in rows:
    print(f"{layer:>5} | {' '.join(repr(t) for t in top):<60} | {_fmt(r_think):>10} | {_fmt(r_say):>10}")

~/repos/interpretune/.claude/worktrees/jlens-research-639/src/interpretune/base/components/mixins.py:102: Analysis configuration has not been set.


layer | top-5 at the answer position                                 | tennis rank | racket rank
    0 | ' b' ' ta' ' pa' ' dea' 'Ta'                                 |          - |          -
    1 | ' b' ' tornado' ' pea' ' great' ' ta'                        |          - |          -
    2 | ' tornado' ' great' ' pea' ' b' ' terrible'                  |          - |          -
    3 | ' great' ' tornado' ' big' ' amerikan' ' bird'               |          - |          -
    4 | ' great' ' big' ' tornado' ' terrible' ' nuclear'            |          - |          -
    5 | ' terrible' ' great' ' big' ' tornado' ' nuclear'            |          - |          -
    6 | ' tornado' ' nuclear' ' terrible' ' boiling' ' great'        |          - |          -
    7 | ' great' ' terrible' ' fantastic' ' amazing' ' tornado'      |          - |          -
    8 | ' amazing' ' fantastic' ' tornado' ' bird' ' terrible'       |          - |          -
    9 | ' amazing' ' mechanical' ' fantastic' ' 

## 4: Where the unstated intermediate is readable

The answer position is where the model *says* Y, but the intermediate X does not have to be read there. This map
marks, for each position and fitted layer, the rank of `THINK_TOKEN` when it is inside the lens top-`READABLE_TOP_K`.

In [7]:
positions = list(range(1, input_ids.shape[1]))  # every prompt position after BOS
think_ranks = {}
for layer, *_ in rows:
    ids_all = torch.as_tensor(read(layer, positions)["jlens_top_token_ids"])[0]
    think_ranks[layer] = [rank_in(ids_all[p], think_id) for p in range(len(positions))]

label_w = max(len(repr(t)) for t in prompt_tokens[1:])
print(" " * (label_w + 1) + " ".join(f"{layer:>3}" for layer in think_ranks))
for p, tok in enumerate(prompt_tokens[1:]):
    marks = []
    for layer in think_ranks:
        r = think_ranks[layer][p]
        marks.append(f"{r:>3}" if r is not None and r < READABLE_TOP_K else "  .")
    print(f"{repr(tok):>{label_w}} " + " ".join(marks))

readable = [
    (layer, p)
    for layer, ranks in think_ranks.items()
    for p, r in enumerate(ranks)
    if r is not None and r < READABLE_TOP_K
]
first_think = min((layer for layer, _ in readable), default=None)
first_say = min((layer for layer, _, _, r in rows if r is not None and r < READABLE_TOP_K), default=None)
print()
print(f"{THINK_TOKEN} first readable (any position): layer {first_think}")
print(f"{SAY_TOKEN} first readable (answer position): layer {first_say}")

               0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23  24
      'Fact'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
         ':'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
      ' The'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
    ' piece'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
       ' of'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
' equipment'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
     ' used'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
       ' to'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .

## 5: Think X versus say Y, side by side

The workspace at the layer and position where the intermediate is best read, against the answer position at the
last fitted layer, where the lens is closest to the model's own output.

In [8]:
if readable:
    best_layer, best_pos = min(readable, key=lambda lp: think_ranks[lp[0]][lp[1]])
    say_layer = rows[-1][0]

    def _top_pairs(out, k=10):
        ids = torch.as_tensor(out["jlens_top_token_ids"])[0, 0][:k]
        scores = torch.as_tensor(out["jlens_top_token_scores"])[0, 0][:k]
        return [(tokenizer.decode([int(t)]), float(s)) for t, s in zip(ids, scores)]

    think_top = _top_pairs(read(best_layer, [positions[best_pos]]))
    say_top = _top_pairs(read(say_layer, [answer_pos]))
    print(f"left: layer {best_layer} at {prompt_tokens[positions[best_pos]]!r}; right: layer {say_layer} at the answer")
    summary = display_jlens_readout_comparison(think_top, say_top, layer=best_layer, basis="jlens_norm_aware")
else:
    print(f"{THINK_TOKEN} is not readable inside the top-{READABLE_TOP_K} at any position or layer for this prompt.")

left: layer 16 at ' ball'; right: layer 24 at the answer


#,Pre token,Pre,Post token,Post,Move
1,tennis,+4480.42,racket,+5325.57,new
2,ball,+4380.50,hammer,+5176.23,new
3,golf,+4379.74,tennis,+5127.29,-2
4,baseball,+4334.00,mallet,+5009.67,new
5,was,+4217.44,wooden,+4923.69,new
6,balls,+3952.39,racquet,+4850.28,new
7,is,+3951.81,paddle,+4543.08,new
8,golfers,+3941.99,shuttle,+4404.62,new
9,golfer,+3876.94,bat,+4225.38,new
10,bowling,+3832.29,club,+4055.99,new


## Reading the result

The verdict below is computed from this run rather than written in advance. When the intermediate is readable at an
earlier layer than the answer, that is the latent-reasoning signature the Jacobian-lens paper describes, read here
through interpretune's `jlens_read` on a published lens. When it is not, the prompt is one of the cases where the
answer surfaces first or at the same depth.

The steering demo (`ct_concept_steering_demo`, sections 4b to 4d) uses the same space to write rather than read.

In [9]:
if first_think is None:
    verdict = f"{THINK_TOKEN} never became readable; this prompt does not show the ordering"
elif first_say is None or first_think < first_say:
    verdict = f"{THINK_TOKEN} was readable at layer {first_think}, before {SAY_TOKEN} (layer {first_say})"
else:
    verdict = f"{SAY_TOKEN} was readable at layer {first_say}, no later than {THINK_TOKEN} (layer {first_think})"
print(verdict)
_stack.close()

tennis was readable at layer 3, before racket (layer 22)
